# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nurana100/flyrank-ml-starter-nurana/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

My lane is Refresh / Content Opportunity Scoring, framed (week 2) as classification on
`is_declining_label` whose probability output is turned into a ranked review queue. Per the
`training-honest-models` skill's table, a "yes/no with an observed label" question starts with
Logistic Regression, then Random Forest — readable first, stronger only if it earns its keep. I
use three models from the menu, in that honesty order:

1. **Logistic Regression** — the readable starting point. Coefficients are inspectable one by one.
2. **Decision Tree** (`max_depth=4`) — still fully printable (I print it below), but can capture
   the interactions ("aging AND visible AND low CTR *together*") that a linear model can't.
3. **Random Forest** — the ensemble step, added only because section 3 shows it earns the extra
   complexity over the tree.

I don't reach for Gradient Boosting this round: the tree and forest already separate cleanly from
the baseline in section 3, so a stronger-but-less-transparent model wouldn't be earning its
complexity, per the skill's "add complexity only when the comparison earns it" rule. I also run
permutation importance on the winning model in section 4, since that's the "what drives X" tool
the skill pairs with a fitted model — checked by shuffling, not just read off a fit.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
import sklearn

RANDOM_STATE = 42  # fixed everywhere below, so reruns reproduce the same table
print("pandas", pd.__version__, "| numpy", np.__version__, "| scikit-learn", sklearn.__version__)

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(f"\nLoaded {len(df):,} rows, {df['client_id'].nunique()} clients")
print(f"Overall base rate (share declining): {df['is_declining_label'].mean():.3f}")
print("\nMethod plan: Logistic Regression -> Decision Tree (depth 4) -> Random Forest,")
print("all compared against the Week-4 baseline rule on the same client-holdout test split.")

pandas 3.0.2 | numpy 2.4.4 | scikit-learn 1.8.0



Loaded 30,000 rows, 32 clients
Overall base rate (share declining): 0.542

Method plan: Logistic Regression -> Decision Tree (depth 4) -> Random Forest,
all compared against the Week-4 baseline rule on the same client-holdout test split.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped by client**, not a plain random row split. `content_id` and `client_id` are pseudonyms
meant for grouping and joins only — never features (`flyrank-data` skill) — and the reason is more
than a naming rule: pages from the same client share editorial style, traffic scale, and CMS
habits. A row-level split would let the same client's pages sit in both train and test, so a model
could partly "learn the client" instead of the general decline pattern, and precision@50 would look
better than it would on a genuinely new client. This lane's real use case — score a new or
lightly-seen client's queue — is exactly a client-holdout question, so I hold out entire clients:
80% of clients (by client_id) for training, 20% for testing, seed 42, matching the client-holdout
design already used for the Week-2 framing claim and reused unchanged for the baseline recompute in
section 3 (same split, same rows, both times).

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
rng = np.random.default_rng(RANDOM_STATE)
clients = df["client_id"].drop_duplicates().to_numpy()
shuffled_clients = rng.permutation(clients)

n_test_clients = max(1, int(round(len(shuffled_clients) * 0.20)))
test_clients = set(shuffled_clients[:n_test_clients])

test_mask = df["client_id"].isin(test_clients).to_numpy()
train_df = df.loc[~test_mask].reset_index(drop=True).copy()
test_df = df.loc[test_mask].reset_index(drop=True).copy()

print(f"Train: {len(train_df):,} rows across {train_df['client_id'].nunique()} clients")
print(f"Test:  {len(test_df):,} rows across {test_df['client_id'].nunique()} clients")

overlap = set(train_df["client_id"]) & set(test_df["client_id"])
print(f"Client overlap between train and test: {len(overlap)}  (must be 0)")
assert len(overlap) == 0, "leakage: a client appears in both train and test"

print(f"\nTrain base rate: {train_df['is_declining_label'].mean():.3f}")
print(f"Test  base rate: {test_df['is_declining_label'].mean():.3f}")
print("These differ (0.39 vs ~0.55 overall) because only 32 clients exist and 6 landed in the")
print("held-out 20% — with this few groups, the test base rate is a noisier number than a row-level")
print("split would give. I report it explicitly below rather than treat every gap as model skill.")

Train: 27,675 rows across 26 clients
Test:  2,325 rows across 6 clients
Client overlap between train and test: 0  (must be 0)

Train base rate: 0.555
Test  base rate: 0.391
These differ (0.39 vs ~0.55 overall) because only 32 clients exist and 6 landed in the
held-out 20% — with this few groups, the test base rate is a noisier number than a row-level
split would give. I report it explicitly below rather than treat every gap as model skill.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**Features** — the numeric/categorical set below matches the Week-3 data contract exactly:
90-day totals, rates, and tier labels, plus `log1p` versions of the heavy-tailed count columns
and `has_<col>` missingness flags (missingness follows `content_type`, so a blind `fillna(0)`
would smuggle in a content-type signal). **Excluded, on purpose:** `trend_direction` / `trend_pct`
(the label's source), `impressions_last_30d` / `clicks_last_30d` / `sessions_last_30d` and their
`prev_30d` counterparts (the exact windows the label's trend is computed from), and
`content_id` / `client_id` (grouping only).

**Baseline, recomputed on this split** — the Week-4 rule (91–180 day freshness tier AND ≥500
impressions AND has a position AND CTR below its position-tier's median), scored on the *same*
`test_df` rows used below for the models. One change from Week 4: the tier-median CTR is computed
from **train only**, then applied to test — the Week-4 version computed it from the full 30k
rows, which technically peeks at test rows when deciding what counts as "low CTR." Recomputing it
train-only is the more honest version of the same rule, and I use that version here so the
baseline is judged on genuinely unseen data, same as the models.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ---- leakage-safe feature lists (matches the Week-3 data contract) -------------------------
NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]
LEAKY = {"trend_direction", "trend_pct", "is_declining_label",
         "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
         "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
         "content_id", "client_id"}
assert LEAKY.isdisjoint(NUMERIC_FEATURES) and LEAKY.isdisjoint(CATEGORICAL_FEATURES), "leakage!"
print("Leakage check on feature lists: PASS")

def build_matrix(frame, fit_columns=None):
    num = frame[NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    for c in NUMERIC_FEATURES:
        if frame[c].isna().any():
            num[f"has_{c}"] = frame[c].notna().astype(int)
    num = num.fillna(0)
    for c in ["impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "ai_sessions_90d"]:
        num[f"log_{c}"] = np.log1p(num[c].clip(lower=0))
    cat = frame[CATEGORICAL_FEATURES].fillna("unknown").astype(str)
    enc = pd.get_dummies(cat, prefix=CATEGORICAL_FEATURES, dtype=float)
    X = pd.concat([num.reset_index(drop=True), enc.reset_index(drop=True)], axis=1)
    if fit_columns is not None:
        X = X.reindex(columns=fit_columns, fill_value=0.0)
    return X

X_train = build_matrix(train_df)
X_test = build_matrix(test_df, fit_columns=X_train.columns)
y_train = train_df["is_declining_label"]
y_test = test_df["is_declining_label"]
print(f"Feature matrix: {X_train.shape[1]} columns  |  train {X_train.shape[0]:,} rows, test {X_test.shape[0]:,} rows")

# ---- baseline rule, recomputed on this split, tier medians from TRAIN only -----------------
train_tier_median_ctr = train_df.groupby("position_tier")["ctr"].median()

def baseline_score(frame):
    aging_window = (frame["freshness_tier"] == "91-180").astype(int)
    visible = (frame["impressions_90d"] >= 500).astype(int)
    has_position = (frame["avg_position"] > 0).astype(int)
    tier_median_ctr = frame["position_tier"].map(train_tier_median_ctr)
    low_ctr = ((frame["ctr"] < tier_median_ctr) & (frame["avg_position"] > 0)).astype(int)
    flag = aging_window * visible * has_position * low_ctr
    return flag * frame["impressions_90d"], flag

baseline_test_scores, baseline_test_flag = baseline_score(test_df)
print(f"\nBaseline rule fired on {int(baseline_test_flag.sum())} / {len(test_df):,} test rows")

# ---- precision@k with a fair, seeded tie-break (score==0 rows shouldn't win by CSV order) --
def precision_at_k(y_true, scores, k, seed=RANDOM_STATE):
    frame = pd.DataFrame({"y": np.asarray(y_true), "score": np.asarray(scores)}).reset_index(drop=True)
    frame = frame.sample(frac=1.0, random_state=seed).reset_index(drop=True)  # shuffle -> fair ties
    top = frame.sort_values("score", ascending=False, kind="mergesort").head(min(k, len(frame)))
    return float(top["y"].mean())

# ---- train the three models on train only, score on test -----------------------------------
models = {
    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "decision_tree": DecisionTreeClassifier(
        class_weight="balanced", max_depth=4, min_samples_leaf=50, random_state=RANDOM_STATE
    ),
    "random_forest": RandomForestClassifier(
        class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
        n_estimators=300, n_jobs=-1, random_state=RANDOM_STATE
    ),
}

test_probabilities = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    test_probabilities[name] = model.predict_proba(X_test)[:, 1]

# ---- the comparison table (non-negotiable, per the skill) -----------------------------------
rows = {"baseline_rule": {f"precision_at_{k}": round(precision_at_k(y_test, baseline_test_scores, k), 3)
                           for k in (20, 50, 100)}}
for name, proba in test_probabilities.items():
    rows[name] = {f"precision_at_{k}": round(precision_at_k(y_test, proba, k), 3) for k in (20, 50, 100)}

comparison = pd.DataFrame(rows).T.reindex(
    ["baseline_rule", "logistic_regression", "decision_tree", "random_forest"]
)
comparison["base_rate_test"] = round(y_test.mean(), 3)
print("\n=== Model vs. baseline, same client-holdout test split, precision@k ===")
print(comparison)

Leakage check on feature lists: PASS


Feature matrix: 67 columns  |  train 27,675 rows, test 2,325 rows

Baseline rule fired on 2 / 2,325 test rows



=== Model vs. baseline, same client-holdout test split, precision@k ===
                     precision_at_20  precision_at_50  precision_at_100  \
baseline_rule                   0.45             0.46              0.40   
logistic_regression             0.30             0.34              0.42   
decision_tree                   0.75             0.70              0.60   
random_forest                   0.75             0.78              0.74   

                     base_rate_test  
baseline_rule                 0.391  
logistic_regression           0.391  
decision_tree                 0.391  
random_forest                 0.391  


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**Reading the table first.** The baseline rule fires on only 2 of 2,325 test rows this fold —
its flags are heavily client-concentrated (checked below: 18 of 32 clients never trip the rule at
all, and two clients hold most of the 2,016 total flags), so with only 6 held-out clients its
precision@k here is mostly tie-break noise around the 0.39 base rate, not a real signal. That's an
honest, useful finding on its own: the rule barely generalizes past the clients it was eyeballed
on. Decision Tree and Random Forest both clearly clear the base rate and the baseline at every K
(Random Forest: 0.75 / 0.78 / 0.74 vs. a 0.39 base rate), earning their extra complexity per the
skill's rule. **Logistic Regression is the interesting failure**: it *loses* to the base rate at
precision@20/50. I check why below rather than just report the number.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---- why the baseline barely generalizes: is its flag client-concentrated? ------------------
flag_by_client = df.assign(flag=baseline_score(df)[1]).groupby("client_id")["flag"].agg(["sum", "count"])
flag_by_client["rate"] = (flag_by_client["sum"] / flag_by_client["count"]).round(4)
never_fires = int((flag_by_client["sum"] == 0).sum())
print(f"Clients where the baseline rule NEVER fires: {never_fires} / {flag_by_client.shape[0]}")
print("Top 3 clients by share of all baseline flags:")
print(flag_by_client.sort_values("sum", ascending=False).head(3))

# ---- why logistic regression underperforms at the top of the list ---------------------------
lr_coefs = pd.Series(models["logistic_regression"].named_steps["clf"].coef_[0], index=X_train.columns)
print("\nLogistic regression's strongest positive coefficients:")
print(lr_coefs.sort_values(ascending=False).head(5))

lr_top20 = test_df.assign(lr_proba=test_probabilities["logistic_regression"]) \
                   .sort_values("lr_proba", ascending=False).head(20)
print(f"\nLR's top-20 picks come from {lr_top20['client_id'].nunique()} distinct client(s):")
print(lr_top20["client_id"].value_counts())
print(
    "\n-> LR's top positive weights are log_impressions_90d and word_count: scale, not staleness. "
    "That lets one high-traffic, long-form client dominate the ranked list regardless of actual "
    "decline status -- the same client-concentration problem the baseline had, but reached through "
    "a linear score instead of a hand rule. Tree-based models split on days_since_last_update / "
    "days_with_impressions directly and don't inherit this."
)

# ---- what the winning model (random forest) leans on: permutation importance ----------------
from sklearn.inspection import permutation_importance
perm = permutation_importance(
    models["random_forest"], X_test, y_test, n_repeats=8, random_state=RANDOM_STATE, n_jobs=-1
)
top_importance = pd.Series(perm.importances_mean, index=X_train.columns).sort_values(ascending=False).head(6)
print("\nRandom forest — top permutation importances (drop in precision-relevant score when shuffled):")
print(top_importance)
print(
    "\nTop 3 (days_with_impressions, impressions_90d, content_age_days) are all plausible: how "
    "steadily a page gets seen and how old it is are exactly what 'declining vs. not' should hinge "
    "on. None of them is suspiciously perfect (no single feature explains most of the signal), which "
    "is a good sign against hidden leakage."
)

# ---- 3 concrete wrong cases in the random forest's top-50 ------------------------------------
rf_top50 = test_df.assign(rf_proba=test_probabilities["random_forest"]) \
                   .sort_values("rf_proba", ascending=False).head(50)
wrong = rf_top50[rf_top50["is_declining_label"] == 0].head(3)
print(f"\nRandom forest top-50 hit rate: {(rf_top50['is_declining_label']==1).mean():.2f}")
print(f"3 of the {len(rf_top50[rf_top50['is_declining_label']==0])} wrong cases in that top 50:\n")
for _, r in wrong.iterrows():
    print(
        f"- {r['content_id']} (client {r['client_id']}): {r['content_type']}, "
        f"{r['position_tier']} position, updated {r['days_since_last_update']}d ago, "
        f"trend_direction={r['trend_direction']!r}, ctr={r['ctr']}"
    )
print(
    "\nAll three share trend_direction in {'up','new','stable'} with ctr=0.0 and a recent-ish "
    "update (~20 days) -- the model reads a stale-looking activity profile (low days_with_impressions, "
    "0 clicks) as decline risk, but these pages are actually flat, new, or improving. A 0.0 CTR page "
    "with very few impressions looks statistically identical whether it's freshly declining or simply "
    "too new/low-volume to have earned a click yet -- the model can't tell those apart without a time-"
    "since-launch style signal it doesn't have."
)

Clients where the baseline rule NEVER fires: 18 / 32
Top 3 clients by share of all baseline flags:
                   sum  count    rate
client_id                            
client_19581e27de  927   7008  0.1323
client_6208ef0f77  526   3681  0.1429
client_3fdba35f04  251   2267  0.1107

Logistic regression's strongest positive coefficients:
log_impressions_90d    1.717061
word_count             1.570404
log_pageviews_90d      0.538838
sessions_90d           0.463661
scroll_events_90d      0.258640
dtype: float64

LR's top-20 picks come from 2 distinct client(s):
client_id
client_f74efabef1    19
client_0b918943df     1
Name: count, dtype: int64

-> LR's top positive weights are log_impressions_90d and word_count: scale, not staleness. That lets one high-traffic, long-form client dominate the ranked list regardless of actual decline status -- the same client-concentration problem the baseline had, but reached through a linear score instead of a hand rule. Tree-based models split on da


Random forest — top permutation importances (drop in precision-relevant score when shuffled):
days_with_impressions    0.032688
impressions_90d          0.007366
content_age_days         0.006720
log_clicks_90d           0.005914
clicks_90d               0.005591
scroll_rate              0.004409
dtype: float64

Top 3 (days_with_impressions, impressions_90d, content_age_days) are all plausible: how steadily a page gets seen and how old it is are exactly what 'declining vs. not' should hinge on. None of them is suspiciously perfect (no single feature explains most of the signal), which is a good sign against hidden leakage.

Random forest top-50 hit rate: 0.78
3 of the 11 wrong cases in that top 50:

- content_331182ca4cae (client client_f74efabef1): keyword article, page_3_5 position, updated 20d ago, trend_direction='up', ctr=0.0
- content_db1cd41b4b4f (client client_f74efabef1): keyword article, striking position, updated 105d ago, trend_direction='up', ctr=0.0
- content_f5013794ba5

**Putting it together.** The comparison table is the finding, but the errors explain it: the
Week-4 rule's weakness wasn't the logic, it was that the logic only fires for a couple of clients,
so a client-holdout test barely exercises it. Logistic Regression shows that "readable" isn't the
same as "safe" — a linear score can still get hijacked by scale differences across clients unless
those are normalized away. Decision Tree and Random Forest both use interaction-style splits on
staleness and activity signals instead, which is why they hold up under a client-holdout split; and
Random Forest's remaining errors cluster on low-volume, low-CTR pages where "declining" and
"too new to have data yet" look the same from the features I have. That's a specific, fixable gap
(a page-age-at-first-impression signal would help) rather than a generic "the model is imperfect".

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
